# Final Assignment: training

This notebook contains the source code for training the weights of the classification model.

## Download dataset

The following cell needs to be ran only once. It downloads and unzips the data. The unzip CLI tool might not be installed by default on Windows, so if you are using windows, please unzip manually into a folder called "data", or see the README.md.

In [1]:
# Downloading the dataset (unzip might not be installed on Windows)
# !curl -L -o ./data.zip https://www.kaggle.com/api/v1/datasets/download/navoneel/brain-mri-images-for-brain-tumor-detection
# !unzip -d ./data ./data.zip

## Importing libraries

See README.md for installing dependencies :).

In [2]:
import os
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, random_split
from torch.optim import Adam
from torchvision import models, datasets, transforms
from sklearn.metrics import balanced_accuracy_score
from tqdm import tqdm

## Setting the seed

Setting the seed for good reproducability :D

In [3]:
torch.manual_seed(21)  # Using my lucky number :D
np.random.seed(21)

## Setting the device used for training

In [4]:
device = "cpu"
if torch.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
print(f"Using: {device}")

Using: cuda


## Preparing the dataset

In [5]:
transform = transforms.Compose(
    [
        transforms.Resize((299, 299)),  # 299 is the minimal image size of InceptionV3
        transforms.ToTensor(),
        transforms.Normalize(0.5, 0.5)
    ]
)

In [6]:
path = os.path.join("data", "brain_tumor_dataset")

In [7]:
dataset = datasets.ImageFolder(
    root=path,
    transform=transform
)
train_dataset, test_dataset = random_split(
    dataset=dataset,
    lengths=[0.8, 0.2]
)

In [8]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

## Creating the model

### Loading pre-trained model

In [9]:
model = models.inception_v3(weights=models.Inception_V3_Weights)

C:\Users\daanw\PycharmProjects\SOW-BKI266\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


### Adding custom classification head

In [10]:
model.fc = nn.Linear(in_features=2048, out_features=2)  # Got the input features from the sourcecode hihi

In [11]:
model = model.to(device)

### Freezing layers

In [12]:
for name, param in model.named_parameters():
    if "Conv2d" in name or "Mixed_5" in name:
        print(f"Froze layer {name}")
        param.requires_grad = False  # Setting requires_grad to False, makes it so the gradient can't be computed, and the optimizers won't update the parameters.

Froze layer Conv2d_1a_3x3.conv.weight
Froze layer Conv2d_1a_3x3.bn.weight
Froze layer Conv2d_1a_3x3.bn.bias
Froze layer Conv2d_2a_3x3.conv.weight
Froze layer Conv2d_2a_3x3.bn.weight
Froze layer Conv2d_2a_3x3.bn.bias
Froze layer Conv2d_2b_3x3.conv.weight
Froze layer Conv2d_2b_3x3.bn.weight
Froze layer Conv2d_2b_3x3.bn.bias
Froze layer Conv2d_3b_1x1.conv.weight
Froze layer Conv2d_3b_1x1.bn.weight
Froze layer Conv2d_3b_1x1.bn.bias
Froze layer Conv2d_4a_3x3.conv.weight
Froze layer Conv2d_4a_3x3.bn.weight
Froze layer Conv2d_4a_3x3.bn.bias
Froze layer Mixed_5b.branch1x1.conv.weight
Froze layer Mixed_5b.branch1x1.bn.weight
Froze layer Mixed_5b.branch1x1.bn.bias
Froze layer Mixed_5b.branch5x5_1.conv.weight
Froze layer Mixed_5b.branch5x5_1.bn.weight
Froze layer Mixed_5b.branch5x5_1.bn.bias
Froze layer Mixed_5b.branch5x5_2.conv.weight
Froze layer Mixed_5b.branch5x5_2.bn.weight
Froze layer Mixed_5b.branch5x5_2.bn.bias
Froze layer Mixed_5b.branch3x3dbl_1.conv.weight
Froze layer Mixed_5b.branch3x3d

## Training the model

### Function for calculating the models accuracy

In [13]:
def balanced_accuracy(model: nn.Module, loader: DataLoader) -> float:
    model.eval()  # Sets model to evaluation mode
    running_balanced_acc = 0

    for X, y in loader:

        X, y = X.to(device), y.to(device)

        with torch.no_grad():
            y_pred = model(X)
            y_pred = torch.argmax(y_pred, dim=-1)

        y, y_pred = y.to("cpu"), y_pred.to("cpu")  # Moving data back to CPU for sklearn

        fraction = len(y_pred) / len(loader.dataset)
        running_balanced_acc += balanced_accuracy_score(y, y_pred) * fraction

    return running_balanced_acc

### Setting hyper parameters

In [14]:
EPOCHS = 50
LR = 1e-6
WEIGHTS_PATH = "weights"

### Initializing optimizer and loss function

In [15]:
optimizer = Adam(params=model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss().to(device)

### Printing initial accuracy

In [16]:
print(f"Balanced test accuracy {balanced_accuracy(model, test_loader) * 100:.2f}%")

Balanced test accuracy 41.14%


### Training Loop

In [17]:
for epoch in range(EPOCHS):

    model.train()  # Setting the model to training mode
    running_loss = 0


    for X, y in tqdm(train_loader, f"Epoch {epoch +1}"):
        X, y = X.to(device), y.to(device)

        y_pred = model(X)[0]
        loss = criterion(y_pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    running_loss /= len(train_loader)
    balanced_acc = balanced_accuracy(model, test_loader)

    path = os.path.join("weights", f"model-{balanced_acc*100:.0f}.pth")
    torch.save(model.state_dict(), path)

    print(f"Epoch complete: {epoch + 1}/{EPOCHS} (loss: {running_loss})")
    print(f"Balanced test accuracy {balanced_acc * 100:.2f}%")

Epoch 1: 100%|██████████| 13/13 [00:02<00:00,  5.38it/s]


Epoch complete: 1/50 (loss: 0.72102513221594)
Balanced test accuracy 44.66%


Epoch 2: 100%|██████████| 13/13 [00:02<00:00,  5.25it/s]


Epoch complete: 2/50 (loss: 0.720584667645968)
Balanced test accuracy 51.95%


Epoch 3: 100%|██████████| 13/13 [00:02<00:00,  5.34it/s]


...

Epoch complete: 48/50 (loss: 0.39271735686522263)
Balanced test accuracy 86.10%


Epoch 49: 100%|██████████| 13/13 [00:02<00:00,  5.54it/s]


Epoch complete: 49/50 (loss: 0.3701163644974048)
Balanced test accuracy 84.10%


Epoch 50: 100%|██████████| 13/13 [00:02<00:00,  5.38it/s]


Epoch complete: 50/50 (loss: 0.38352733850479126)
Balanced test accuracy 84.10%
